This notebook produces data on clue guesses using self-consistency. 
Right now, you can determine 
- CLUE_LENGTH (the desired length of the clue solution)
- NUM_FILLED (the number of random filled-in letters in the pattern given to the LLM)
- NUM_CLUES (number of clues processed)
- DAY_OF_WEEK

In the future, we should also consider 
- cut-off dates for the puzzle clues used
- self-consistency or no?
    - self consistency parameters as well (num samples = 3, max guesses = 5)

In [1]:
import os
import random
import pandas as pd
from dataclasses import dataclass, field

from src.prompts.get_guesses_with_self_consistency import get_guesses_with_self_consistency
from src.classes.clue import Clue

### **Set Constants and Validate Inputs**

In [2]:
# Params
CLUE_LENGTH = 5
NUM_FILLED = 0       # Must be less than CLUE_LENGTH
NUM_CLUES = 10       # Must be an int or "All"
DAY_OF_WEEK = "Monday"  # "All", "Monday", ..., "Sunday"

# Ignore these
VALID_DAYS = {"All", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"}
TOP_N_GUESSES = 5

# Paths
INPUT_CSV = "crossword_clues.csv"
OUTPUT_DIR = "data/get_guesses_per_clue"

@dataclass
class ExperimentConfig:
    clue_length: int
    num_filled: int
    num_clues: int | str
    day_of_week: str

    def validate(self) -> None:
        if self.day_of_week not in VALID_DAYS:
            raise ValueError(f"Invalid DAY_OF_WEEK '{self.day_of_week}'. Must be one of: {VALID_DAYS}")
        if self.num_filled >= self.clue_length:
            raise ValueError("NUM_FILLED must be strictly less than CLUE_LENGTH.")
        if not isinstance(self.num_clues, int) and self.num_clues != "All":
            raise ValueError("NUM_CLUES must be an integer or 'All'.")

    @property
    def output_path(self) -> str:
        return os.path.join(OUTPUT_DIR, f"test_clues_{self.num_clues}_{self.day_of_week}_fill_{self.num_filled}_length_{self.clue_length}.csv")

config = ExperimentConfig(
    clue_length=CLUE_LENGTH,
    num_filled=NUM_FILLED,
    num_clues=NUM_CLUES,
    day_of_week=DAY_OF_WEEK,
)
config.validate()

## **Read in Crossword Clue CSV Data, Apply Filters**

In [3]:
# read in crossword clue csv data
data = pd.read_csv(INPUT_CSV)
print(f"Loaded dataset: {data.shape}")
display(data.head())

# get clues with the specified length (CLUE_LENGTH)
clues = data[data["length"] == config.clue_length]

# filter by day of week (DAY_OF_WEEK)
if config.day_of_week != "All":
    clues = clues[clues["day_of_week"] == config.day_of_week]

# get specified number of clues (NUM_CLUES)
if isinstance(config.num_clues, int):
    if config.num_clues > len(clues):
        raise ValueError(
            f"Requested {config.num_clues} clues, but only {len(clues)} match your criteria."
        )
    clues = clues.iloc[: config.num_clues]

print(f"Filtered clues: {clues.shape}")
display(clues.head())

Loaded dataset: (23143, 8)


,file,date,day_of_week,number,direction,length,text,solution
0,puz_files/nytm_2015_01_01.puz,2015-01-01,Thursday,1,across,4,"2015, in Roman numerals",MMXV
1,puz_files/nytm_2015_01_01.puz,2015-01-01,Thursday,4,across,3,"""Spare"" thing at a barbecue",RIB
2,puz_files/nytm_2015_01_01.puz,2015-01-01,Thursday,6,across,5,Heart beater in bridge bidding,SPADE
3,puz_files/nytm_2015_01_01.puz,2015-01-01,Thursday,8,across,3,Delivery from Santa,TOY
4,puz_files/nytm_2015_01_01.puz,2015-01-01,Thursday,9,across,4,Channel for armchair athletes,ESPN


Filtered clues: (10, 8)


,file,date,day_of_week,number,direction,length,text,solution
42,puz_files/nytm_2015_01_05.puz,2015-01-05,Monday,6,across,5,Second-___ (question after the fact),GUESS
46,puz_files/nytm_2015_01_05.puz,2015-01-05,Monday,2,down,5,Teakettle emission,STEAM
110,puz_files/nytm_2015_01_12.puz,2015-01-12,Monday,1,across,5,Covers with a cold blanket?,SNOWS
111,puz_files/nytm_2015_01_12.puz,2015-01-12,Monday,6,across,5,2014 movie based on a Broadway musical,ANNIE
112,puz_files/nytm_2015_01_12.puz,2015-01-12,Monday,7,across,5,Hunky-dory,SWELL


## **Get guesses, without the board information**

5 CSVs, 
1) 0 letters given
2) 1 letter given
3) 2 letters given
4) 3 letters given
5) 4 letters given

Each CSV should include the following columns: 

These are included in the crossword_clues.csv file:
- file
- date
- day_of_week
- number
- direction
- length
- text
- solution

New columns: 

- pattern_given (i.e., _ _ N _ _ )

- guess1
- guess2
- guess3
- guess4
- guess5

- confidence1
- confidence2
- confidence3
- confidence4
- confidence5


In [4]:
# Return a letter pattern with NUM_FILLED letters revealed at random
def build_pattern(solution: str, num_filled: int) -> list[str]:
    pattern = ["_"] * len(solution)
    fill_positions = random.sample(range(len(solution) - 1), num_filled)
    for pos in fill_positions:
        pattern[pos] = solution[pos]
    return pattern


@dataclass
class GuessResult:
    guesses: list[str] = field(default_factory=lambda: [""] * TOP_N_GUESSES)
    confidences: list[float] = field(default_factory=lambda: [0.0] * TOP_N_GUESSES)
    top1_correct: bool = False
    top5_correct: bool = False


def evaluate_clue(clue: Clue, pattern: list[str], solution: str) -> GuessResult:
    result = GuessResult()
    clue_guesses = get_guesses_with_self_consistency(clue, pattern, filter=False, debug=True)
    for i, guess in enumerate(clue_guesses[:TOP_N_GUESSES]):
        result.guesses[i] = guess.answer
        result.confidences[i] = guess.confidence_score
    result.top5_correct = solution in result.guesses
    result.top1_correct = result.guesses[0] == solution
    return result


RESULT_COLUMNS = [
    "file", "date", "day_of_week", "number", "direction", "length",
    "text", "solution", "pattern",
    *[f"guess{i}" for i in range(1, TOP_N_GUESSES + 1)],
    *[f"confidence{i}" for i in range(1, TOP_N_GUESSES + 1)],
]

def build_result_row(row: pd.Series, solution: str, pattern: list[str], result: GuessResult) -> list:
    return [
        row["file"], row["date"], row["day_of_week"], row["number"],
        row["direction"], row["length"], row["text"], solution,
        " ".join(pattern),
        *result.guesses,
        *result.confidences,
    ]

In [ ]:
results = []
top1_success = 0
top5_success = 0

# for every clue given
for _, row in clues.iterrows():
    print(row)
    clue = Clue(
        text=row["text"],
        length=row["length"],
        number=row["number"],
        direction=row["direction"],
    )
    solution = row["solution"]
    pattern = build_pattern(solution, config.num_filled)
    result = evaluate_clue(clue, pattern, solution)

    if result.top1_correct:
        top1_success += 1
    if result.top5_correct:
        top5_success += 1

    results.append(build_result_row(row, solution, pattern, result))


results_df = pd.DataFrame(results, columns=RESULT_COLUMNS)
display(results_df.head())

os.makedirs(os.path.dirname(config.output_path), exist_ok=True)
results_df.to_csv(config.output_path, index=False)

total = len(clues)
print("Successfully wrote results to CSV file.")
print(f"Top 1 success rate: {top1_success}/{len(clues)}")
print(f"Top 5 success rate: {top5_success}/{len(clues)}")

file                  puz_files/nytm_2015_01_05.puz
date                                     2015-01-05
day_of_week                                  Monday
number                                            6
direction                                    across
length                                            5
text           Second-___ (question after the fact)
solution                                      GUESS
Name: 42, dtype: object
  [Self-consistency] Sample 1/3...
=== GENERATE GUESSES with llama3.1:latest ===
You are a crossword expert. Provide up to five different guesses that fit the clue and pattern.
Constraints:
- Clue: Second-___ (question after the fact)
- Length: 5 letters

 - The first letter is unknown.
 - The second letter is unknown.
 - The third letter is unknown.
 - The fourth letter is unknown.
 - The fifth letter is unknown.

Final Output:
Return only your guesses in ALL CAPS, a confidence score (0-100), and an explanation. Every letter used must be in the English 